# Tester performance de nos méthodes 

In [25]:
import pandas as pd
import os, glob
import numpy as np
from collections import defaultdict
import json
import cv2
# import glob, os
import matplotlib.pyplot as plt
from skimage import io
# from scipy.ndimage import median_filter
from scipy.ndimage import convolve1d

### Les fonctions de nos méthodes 

#### Pour fusionner les segments trouvés 

In [36]:
from collections import defaultdict

def fusionner_segments(segments_etendus):
    """
    Prend une liste de segments sous la forme [x, y_start, y_end],
    les regroupe par colonne (x), puis fusionne les segments qui se chevauchent ou se touchent.
    """
    if not segments_etendus:
        return {}

    # 1. Regrouper les segments par colonne (x)
    dict_intermediaire = defaultdict(list)
    for segment in segments_etendus:
        # Ici on lit simplement les 3 valeurs de ta liste [x, y_start, y_end]
        x = segment[0]
        y_start = segment[1]
        y_end = segment[2]
        dict_intermediaire[x].append((y_start, y_end))

    dict_final = {}

    # 2. Traiter chaque colonne indépendamment
    for x, liste_segments in dict_intermediaire.items():
        
        if len(liste_segments) <= 1:
            dict_final[x] = liste_segments
            continue
            
        segments_tries = sorted(liste_segments, key=lambda coord: coord[0])
        segments_fusionnes = [segments_tries[0]]
        
        for segment_actuel in segments_tries[1:]:
            dernier_segment_valide = segments_fusionnes[-1]
            
            debut_actuel, fin_actuelle = segment_actuel
            debut_dernier, fin_derniere = dernier_segment_valide
            
            if debut_actuel <= fin_derniere:
                nouvelle_fin = max(fin_derniere, fin_actuelle)
                segments_fusionnes[-1] = (debut_dernier, nouvelle_fin)
            else:
                segments_fusionnes.append(segment_actuel)
                
        dict_final[x] = segments_fusionnes

    return dict_final

In [27]:
def fusionner_segments(segments):
    """
    Fonction utilitaire : si deux petits défauts s'étendent et finissent 
    par se rentrer dedans, on les fusionne en un seul grand défaut.
    """
    if not segments: return []
    
    # Grouper par colonne (x)
    dict_cols = {}
    for seg in segments:
        x, y_s, y_e = seg[0][0], seg[0][1], seg[1][1]
        if x not in dict_cols: dict_cols[x] = []
        dict_cols[x].append([y_s, y_e])
        
    segments_fusion = []
    for x, intervalles in dict_cols.items():
        intervalles.sort(key=lambda v: v[0]) # Trier de haut en bas
        fusion = [intervalles[0]]
        
        for courant in intervalles[1:]:
            dernier = fusion[-1]
            # Si les segments se chevauchent ou se touchent
            if courant[0] <= dernier[1] + 1:
                dernier[1] = max(dernier[1], courant[1])
            else:
                fusion.append(courant)
                
        for y_s, y_e in fusion:
            segments_fusion.append([(x, y_s), (x, y_e)])
            
    return segments_fusion

#### CFAR_band + Region Growing

In [28]:
def detecter_et_mesurer_defauts_complet(image, hauteur_bande=50, train_cells=4, guard_cells=2, multiplicateur_rupture=3.5):
    """
    Pipeline complet de détection et mesure des colonnes défectueuses (entières et fragmentées).
    
    1. Découpe l'image en bandes et applique un filtre CA-CFAR pour trouver des "graines" de défauts.
    2. Prolonge ces graines vers le haut et le bas (Region Growing) jusqu'à une rupture d'intensité.
    3. Fusionne les segments qui se chevauchent et formate le résultat final.
    
    Args:
        image (numpy.ndarray): L'image 16-bits en entrée.
        hauteur_bande (int): Taille de la bande horizontale pour le CFAR (défaut: 50).
        train_cells (int): Nombre de cellules d'entraînement de chaque côté (défaut: 4).
        guard_cells (int): Nombre de cellules de garde de chaque côté (défaut: 2).
        multiplicateur_rupture (float): Tolérance pour le seuil de rupture lors de la croissance (défaut: 3.5).
        
    Returns:
        dict: Dictionnaire formaté {x: [(y1_start, y1_end), (y2_start, y2_end), ...]} 

    """
    height, width = image.shape

    # =========================================================
    # ETAPE 1 : DETECTION DES GRAINES (CFAR par bandes)
    # =========================================================
    segments_initiaux = []
    
    # --- Paramètres du CFAR ---
    num_train_side = train_cells 
    num_guard_side = guard_cells  
    taille_fenetre = (num_train_side * 2) + (num_guard_side * 2) + 1 
    noyau = np.zeros(taille_fenetre)
    noyau[:num_train_side] = 1.0  
    noyau[-num_train_side:] = 1.0 
    noyau = noyau / (num_train_side * 2)

    # On parcourt l'image de haut en bas, en sautant de 'hauteur_bande' en 'hauteur_bande'
    for y_start in range(0, height, hauteur_bande):
        y_end = min(y_start + hauteur_bande, height) # min() pour ne pas déborder à la fin
        
        # On extrait la sous-image (la bande horizontale)
        bande = image[y_start:y_end, :]
        
        # On calcule la projection médiane UNIQUEMENT sur cette bande
        projection_bande = np.median(bande, axis=0)
        
        # On applique le CFAR
        bruit_de_fond_local = convolve1d(projection_bande, noyau, mode='nearest')
        
        # Tolérance locale pour cette bande spécifique
        ecart_type_bande = np.std(projection_bande)
        tolerance = 3 * ecart_type_bande  
        
        seuil_haut = bruit_de_fond_local + tolerance
        seuil_bas = bruit_de_fond_local - tolerance
        
        # Détection pour cette bande
        colonnes_detectees = np.where(
            (projection_bande > seuil_haut) | 
            (projection_bande < seuil_bas)
        )[0]
        
        # On enregistre les résultats sous forme de segments initiaux (graines)
        for x in colonnes_detectees:
            segments_initiaux.append([(x, y_start), (x, y_end)])


    # =========================================================
    # ETAPE 2 : CROISSANCE DE REGION (Region Growing)
    # =========================================================
    segments_etendus = []

    for segment in segments_initiaux:
        # On force la conversion en entier natif python pour la suite
        x = int(segment[0][0])
        y_start = int(segment[0][1])
        y_end = int(segment[1][1])
        
        colonne = image[:, x].astype(np.float32) # pour chaque x, on prend tous les px de la col
        
        # 1. Calculer ce qu'est un "saut normal" / et seuil sur cette colonne
        sauts_verticaux = np.abs(np.diff(colonne))  # On regarde la dérivée absolue (la différence entre chaque pixel et le suivant (de la col))

        bruit_normal = np.median(sauts_verticaux) # on calc les sauts normaux (sur le défaut, ou sur le vrai paysage de l'img) en prennant médiane de tous les sauts 
        ecart_sauts = np.std(sauts_verticaux)
        
        seuil_rupture = bruit_normal + (multiplicateur_rupture * ecart_sauts) # calc marche qu'on considère trop grande (fin du défaut)

        # 2. Prolonger vers le HAUT (on remonte la colonne)
        while y_start > 0: # boucle tant qu'on a pas atteint le haut de l'image
            # Quelle est la taille de la marche pour monter sur le pixel du dessus ?
            saut_haut = np.abs(colonne[y_start] - colonne[y_start - 1]) # diff entre px et celui d'avnat 
            
            if saut_haut < seuil_rupture:
                # Intensité similaire : on est toujours dans le défaut
                y_start -= 1 # on continue de rémonter 
            else:
                # BOUM ! Rupture forte d'intensité, on a trouvé le bord supérieur.
                break

        # 3. Prolonger vers le BAS (on descend la colonne)
        while y_end < (height - 1): # boucle tant qu'on a pas atteint le bas de l'image
            # Quelle est la taille de la marche pour descendre sur le pixel du dessous ?
            saut_bas = np.abs(colonne[y_end] - colonne[y_end + 1]) # diff entre px et celui d'après 
            
            if saut_bas < seuil_rupture:
                y_end += 1 # on continue de descendre 
            else:
                break # sinon saut trop gros, on a trouvé bord inf
                
        # On stocke les coordonnées étendues sous forme de liste simple pour la fusion
        segments_etendus.append([x, y_start, y_end])

          
    return fusionner_segments(segments_etendus) # ne pas oublier de fusionner au cas où y'a des chevauchements


#### Local Thresholding with Variance, Median or Mean 

In [29]:
def local_threshold(image, taille_fenetre=50, metrique ='Moyenne', facteur_std=1.0,visual_graph = False):

    height = image.shape[0]

    if metrique == 'Moyenne':
        metrique_ = np.mean(image, axis=0)
        label = 'Moyenne'
    elif metrique == 'Mediane':
        metrique_ = np.median(image, axis=0)
        label = 'Mediane'
    elif metrique == 'Variance':
        metrique_ = np.std(image, axis=0)
        label = 'Variance'
    else : 
        print('Error de metrique')

    filtre = np.ones(taille_fenetre) / taille_fenetre
    tendance_locale = convolve1d(metrique_, filtre, mode='reflect')
    
    #Marge de tolérance
    marge_tolerance = facteur_std * np.std(metrique_)
    
    # Le seuil n'est plus un simple nombre, c'est un tableau de la même taille que l'image !
    threshold_haut = tendance_locale + marge_tolerance
    threshold_bas = tendance_locale - marge_tolerance
    
    # Détection bilatérale avec l'écart absolu (np.abs)
    defect_columns = np.where(np.abs(metrique_ - tendance_locale) > marge_tolerance)[0]

    predit_dict = {}
    for col in defect_columns:
        # On convertit 'col' en int natif Python pour éviter les soucis avec JSON/Dictionnaires
        # Et on indique que le défaut s'étend de la ligne 0 jusqu'en bas (height)
        predit_dict[int(col)] = [(0, height)]

    if visual_graph:
        # Printing results    
        print(f"Analyse basée sur : {label}")
        print(f"Marge de tolérance (+ 2*std) : {marge_tolerance:.2f}")
        print(f"Nombre de colonnes dépassant le seuil local : {len(defect_columns)}")
        print(f"Index de ces colonnes : {defect_columns}")

        # Affichage du graphe
        plt.figure(figsize=(12, 6))
        
        # Tracer la courbe de toutes les variances/moyennes
        plt.plot(metrique_, label=label, color='#1f77b4', linewidth=1.5, alpha=0.8)
        
        # Tracer la courbe du threshold local
        plt.plot(threshold_haut, color='orange', linestyle='--', linewidth=2, label='Seuil Haut (+2 std)')
        plt.plot(threshold_bas, color='green', linestyle='--', linewidth=2, label='Seuil Bas (-2 std)')
        
        # Mettre un point rouge sur le graphe pour chaque "pire" colonne
        plt.scatter(defect_columns, metrique_[defect_columns], color='red', zorder=5, label='Colonnes critiques')

        # Personnalisation du graphe
        plt.title(f'{label} des intensités par colonne (avec Seuil Local)', fontsize=14)
        plt.xlabel('Index de la colonne (Position X dans l\'image)', fontsize=12)
        plt.ylabel(f'{label}', fontsize=12)
        plt.legend()
        plt.grid(True, linestyle=':', alpha=0.7)
        
        plt.tight_layout()
        plt.show()
    return predit_dict

### Local thresholding par bandes 

In [30]:
from collections import defaultdict

def local_threshold_bandes(image, num_bandes=4, taille_fenetre=50, metrique='Moyenne', facteur_std=1.0, visual_graph=False):
    height, width = image.shape
    
    # On calcule la hauteur d'une bande selon le nb que y'en a 
    hauteur_bande = height // num_bandes
    
    # defaultdict permet d'ajouter facilement des tuples à une liste existante
    predit_dict = defaultdict(list)
    
    # Stockage pour le graphe si demandé
    debug_data = []

    for b in range(num_bandes):
        # 1. Définir les limites y de la bande
        y_start = b * hauteur_bande
        # Si c'est la dernière bande, on va jusqu'au bout de l'image (pour ne pas rater de pixels à cause de l'arrondi)
        y_end = height if b == num_bandes - 1 else (b + 1) * hauteur_bande
        
        # 2. Extraire la bande
        bande = image[y_start:y_end, :]

        # 3. Calcul de la métrique sur CETTE bande
        if metrique == 'Moyenne':
            metrique_ = np.mean(bande, axis=0)
            label = 'Moyenne'
        elif metrique == 'Mediane':
            metrique_ = np.median(bande, axis=0)
            label = 'Mediane'
        elif metrique == 'Variance':
            metrique_ = np.std(bande, axis=0)
            label = 'Variance'
        else: 
            print('Erreur de métrique')
            return {}

        # 4. Lissage (Tendance locale)
        filtre = np.ones(taille_fenetre) / taille_fenetre
        tendance_locale = convolve1d(metrique_, filtre, mode='reflect')
        
        # 5. Marge de tolérance (ajustée avec facteur_std !)
        marge_tolerance = np.std(metrique_) * facteur_std
        
        # 6. Détection bilatérale
        defect_columns = np.where(np.abs(metrique_ - tendance_locale) > marge_tolerance)[0]

        # 7. Enregistrement dans le dictionnaire
        for col in defect_columns:
            # On stocke les coordonnées (y_start, y_end) spécifiques à cette bande
            predit_dict[int(col)].append((y_start, y_end))
            
        if visual_graph:
            debug_data.append((b, metrique_, tendance_locale, marge_tolerance, defect_columns))

    if visual_graph:
        # On crée des sous-graphes pour chaque bande
        fig, axes = plt.subplots(num_bandes, 1, figsize=(12, 3 * num_bandes), sharex=True)
        if num_bandes == 1:
            axes = [axes] # Sécurité si on ne demande qu'une seule bande
            
        fig.suptitle(f'Analyse par bandes basée sur : {label} (Fenêtre={taille_fenetre}, Std={facteur_std})', fontsize=16)

        for ax, data in zip(axes, debug_data):
            b, met_, tend_, marge_, def_cols = data
            
            ax.plot(met_, label=f'{label} Bande {b+1}', color='#1f77b4', linewidth=1.5, alpha=0.8)
            ax.plot(tend_ + marge_, color='orange', linestyle='--', linewidth=1.5, label=f'+{facteur_std} std')
            ax.plot(tend_ - marge_, color='green', linestyle='--', linewidth=1.5, label=f'-{facteur_std} std')
            ax.scatter(def_cols, met_[def_cols], color='red', zorder=5)
            
            ax.set_ylabel(f'Bande {b+1}')
            ax.grid(True, linestyle=':', alpha=0.7)
            ax.legend(loc='upper right')
            
        axes[-1].set_xlabel('Index de la colonne (Position X dans l\'image)')
        plt.tight_layout()
        plt.show()

    # On convertit le defaultdict en dict classique avant de le renvoyer
    return dict(predit_dict)

### Notre fonction d'évéluation de perf

#### Pour load un dossier, le JSON, et recup les défauts gt pour une certaine image

In [31]:
def load_images(folder='train',type='VGA',sequence='sequence_1', dyn='low dyn with columns 1', force_gray=False):
    """
    function that load a folder of images

    args : 
    folder : folder type
    type : type of the frame (HD, VGA, SXGA)
    sequence : sequence_1, sequence_2, sequence_3
    dyn : low dyn with columns 1, low dyn with columns 2, low dyn with columns 3
    force_gray : bool to load in gray (one array)

    return :
    list of the images of the folder
    """
    images = []

    chemin_recherche = os.path.join(folder, type, sequence, dyn, '*.png')

    fichiers_trouves = sorted(glob.glob(chemin_recherche))

    if force_gray == True:
        for image_path in fichiers_trouves:
            img = io.imread(image_path, as_gray=True)
            images.append(img)
    else:
        for image_path in fichiers_trouves:
            img = io.imread(image_path)
            images.append(img)
            
    return images


def load_json(file_path):
    """
    Loads a JSON file and returns its content.

    Args:
        file_path (str): The path to the JSON file.

    Returns:
        list/dict: The parsed JSON data.
    """
    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)
    
    return data


def get_defect_coordinates(json_data, image_number):
    """
    Parses JSON data to extract defect coordinates for a specific image number.

    Args:
        json_data (list): The list of dictionaries loaded from the JSON file.
        image_number (str or int): The specific image number to look for in the JSON.

    Returns:
        dict: A dictionary where keys are x-coordinates (columns) and values 
              are dictionaries containing lists of 'start' and 'stop' y-coordinates.
    """
    defect_dict = {}
    
    for item in json_data:
        value = item.get('signal', {}).get(str(image_number))
        
        if value is not None:
            for x_coord in item.get('x_coord', []):
                
                defect_dict[x_coord] = {
                    'ycords': (list(zip(item.get('y_start', []),item.get('y_stop', [])))),
                    'type': item.get('name', [])
                }

    return defect_dict

#### L'évaluation : comparaison des dico gt/res image par image 

In [32]:
def evaluate_detection(vrai_dict, predit_dict, printing = False):
    """
    Évalue la détection avec correspondance exacte (sans tolérance).
    Force la conversion des clés en entiers (int) pour éviter les erreurs de type string/int.
    Calcule et retourne la Précision, le Rappel et le F1-Score.
    """
    vrais_positifs = 0
    faux_negatifs = 0
    faux_positifs = 0
    
    # On convertit toutes les colonnes prédites en int pour être sûr du format
    colonnes_predites_int = {int(x) for x in predit_dict.keys()}
    colonnes_predites_utilisees = set()

    # =========================================================
    # 1. Vérifier ce qui est bien détecté et ce qui est oublié
    # =========================================================
    for x_vrai_raw, infos_vrai in vrai_dict.items():
        x_vrai = int(x_vrai_raw) # On force la vérité terrain en int
        
        # On vérifie la correspondance exacte
        if x_vrai in colonnes_predites_int:
            vrais_positifs += 1
            colonnes_predites_utilisees.add(x_vrai)
        else:
            faux_negatifs += 1

    # =========================================================
    # 2. Vérifier les fausses alarmes (Faux Positifs)
    # =========================================================
    for x_pred in colonnes_predites_int:
        # Si la colonne prédite n'a pas matché avec une vraie colonne
        if x_pred not in colonnes_predites_utilisees:
            faux_positifs += 1

    # =========================================================
    # 3. Calcul des statistiques finales
    # =========================================================
    precision = vrais_positifs / (vrais_positifs + faux_positifs) if (vrais_positifs + faux_positifs) > 0 else 0
    rappel = vrais_positifs / (vrais_positifs + faux_negatifs) if (vrais_positifs + faux_negatifs) > 0 else 0
    
    # Calcul du F1-Score
    if (precision + rappel) > 0:
        f1_score = 2 * (precision * rappel) / (precision + rappel)
    else:
        f1_score = 0
    
    if printing:
        print("\n--- RÉSULTATS DE LA DÉTECTION (Correspondance exacte) ---")
        print(f"Vrais Positifs (Bien trouvés)  : {vrais_positifs}")
        print(f"Faux Négatifs (Oubliés)        : {faux_negatifs}")
        print(f"Faux Positifs (Fausses alarmes): {faux_positifs}")
        print(f"Précision (Fiabilité)          : {precision*100:.1f}%")
        print(f"Rappel (Taux de découverte)    : {rappel*100:.1f}%")
        print(f"F1-Score (Score global)        : {f1_score*100:.1f}%")

    return vrais_positifs, faux_positifs, faux_negatifs, f1_score

#### L'évaluation de la methode sur toute une séquence 

In [ ]:
def evaluate_sequence_detection(methode_detection=local_threshold, 
                                folder='train', img_type='VGA', sequence='sequence_1', dyn='low dyn with columns 1', update=False):
    """
    Évalue la détection sur toute une séquence d'images.
    dataset_name : Le nom du dataset (qui servira à trouver les images et le JSON)
    methode_detection : La fonction à utiliser pour prédire les défauts
    """
    # 1. Charger les images (assure-toi que load_images gère bien le format attendu)

    chiffre = int(dyn.split()[-1])

    data = load_images(folder=folder, type=img_type, sequence=sequence, dyn=dyn, force_gray=False)
    json_name = f'{img_type}_{sequence}_config_{chiffre}'
    
    # recup les JSON 
    chemin_json = f'results/{json_name}.json'
    json_data = load_json(chemin_json)
    
    # Initialisation des compteurs GLOBAUX
    total_vp = 0 # true positive
    total_fp = 0 # false positive
    total_fn = 0 # false negative

    # 3. Boucle sur chaque image
    for i in range(len(data)):
        vrai_dict = get_defect_coordinates(json_data, i) # on recup dico des vrais defauts gt
        predit_dict = methode_detection(data[i]) # on recup dico des défauts qu'on trouve avec notre methode
        vp, fp, fn, f1_image = evaluate_detection(vrai_dict, predit_dict) # on compare les 2 dicos 
                                                                        # possible d'afficher les res de la detection à chaque image (ATTENTION peut polluer le terminal si trop d'images)
                                                                        # en mettant printing = True 
        # On ajoute aux compteurs globaux
        total_vp += vp 
        total_fp += fp
        total_fn += fn

    # =========================================================
    # 4. Calcul des scores globaux de la séquence
    # =========================================================
    precision_seq = total_vp / (total_vp + total_fp) if (total_vp + total_fp) > 0 else 0
    rappel_seq = total_vp / (total_vp + total_fn) if (total_vp + total_fn) > 0 else 0
    
    if (precision_seq + rappel_seq) > 0:
        f1_score_seq = 2 * (precision_seq * rappel_seq) / (precision_seq + rappel_seq)
    else:
        f1_score_seq = 0

    # 5. Affichage des résultats
    if update is True : 
        print("\n" + "="*50)
        print(f"RÉSULTATS GLOBAUX - SÉQUENCE : type : {img_type} sequence : {sequence} dyn : {dyn}")
        print("="*50)
        print(f"Total Vrais Positifs (VP) : {total_vp}")
        print(f"Total Faux Négatifs (FN)  : {total_fn} (Oublis)")
        print(f"Total Faux Positifs (FP)  : {total_fp} (Fausses alarmes)")
        print("-" * 50)
        print(f"Précision de la séquence  = {precision_seq*100:.2f}%")
        print(f"Recall de la séquence     = {rappel_seq*100:.2f}%")
        print(f"F1_score de la séquence   = {f1_score_seq*100:.2f}%")
        print("="*50 + "\n")
    
    return precision_seq, rappel_seq, f1_score_seq

## Test des différentes méthode avec notre fctn d'évaluation de perf

In [34]:
from functools import partial

In [37]:
print("Test de Local Thresholding avec la Métrique de l'Ecart-Type")
evaluate_sequence_detection(
    methode_detection= partial(local_threshold, metrique='Variance'),
    folder="train",
    img_type="VGA",
    sequence="sequence_1",
    dyn="low dyn with columns 3"
)
print(50*"==")
print("Test de Local Thresholding avec la Métrique de la Médiane")
evaluate_sequence_detection(
    methode_detection=partial(local_threshold, metrique='Mediane'),
    folder="train",
    img_type="VGA",
    sequence="sequence_1",
    dyn="low dyn with columns 3"
)
print(50*"==")
print("Test de Local Thresholding avec la Métrique de la Moyenne")
evaluate_sequence_detection(
    methode_detection= partial (local_threshold, metrique='Moyenne'),
    folder="train",
    img_type="VGA",
    sequence="sequence_1",
    dyn="low dyn with columns 3"
)
print(50*"==")
print("Test de CFAR_band et Region Growing")
evaluate_sequence_detection(
    methode_detection= detecter_et_mesurer_defauts_complet,
    folder="train",
    img_type="VGA",
    sequence="sequence_1",
    dyn="low dyn with columns 3"
)

Test de Local Thresholding avec la Métrique de l'Ecart-Type

RÉSULTATS GLOBAUX - SÉQUENCE : type : VGA sequence : sequence_1 dyn : low dyn with columns 3
Total Vrais Positifs (VP) : 1513
Total Faux Négatifs (FN)  : 1454 (Oublis)
Total Faux Positifs (FP)  : 53292 (Fausses alarmes)
--------------------------------------------------
Précision de la séquence  = 2.76%
Recall de la séquence     = 50.99%
F1_score de la séquence   = 5.24%

Test de Local Thresholding avec la Métrique de la Médiane

RÉSULTATS GLOBAUX - SÉQUENCE : type : VGA sequence : sequence_1 dyn : low dyn with columns 3
Total Vrais Positifs (VP) : 2582
Total Faux Négatifs (FN)  : 385 (Oublis)
Total Faux Positifs (FP)  : 0 (Fausses alarmes)
--------------------------------------------------
Précision de la séquence  = 100.00%
Recall de la séquence     = 87.02%
F1_score de la séquence   = 93.06%

Test de Local Thresholding avec la Métrique de la Moyenne

RÉSULTATS GLOBAUX - SÉQUENCE : type : VGA sequence : sequence_1 dyn : low

(0.9247538677918424, 0.886417256488035, 0.905179831354328)

#### Test du Local Thresholding par bandes pour Var, Mean, Median

In [40]:
print(50*"==")
print("Test de Local Thresholding par bandes avec la Métrique Variance")
evaluate_sequence_detection(
    methode_detection= partial(local_threshold_bandes, metrique='Variance'),
    folder="train",
    img_type="VGA",
    sequence="sequence_1",
    dyn="low dyn with columns 3"
)
print(50*"==")
print("Test de Local Thresholding par bandes avec la Métrique Mediane")
evaluate_sequence_detection(
    methode_detection= partial(local_threshold_bandes, metrique='Mediane'),
    folder="train",
    img_type="VGA",
    sequence="sequence_1",
    dyn="low dyn with columns 3"
)
print(50*"==")
print("Test de Local Thresholding par bandes avec la Métrique Moyenne")
evaluate_sequence_detection(
    methode_detection= partial(local_threshold_bandes, metrique='Moyenne'),
    folder="train",
    img_type="VGA",
    sequence="sequence_1",
    dyn="low dyn with columns 3"
)

Test de Local Thresholding par bandes avec la Métrique Variance

RÉSULTATS GLOBAUX - SÉQUENCE : type : VGA sequence : sequence_1 dyn : low dyn with columns 3
Total Vrais Positifs (VP) : 2395
Total Faux Négatifs (FN)  : 572 (Oublis)
Total Faux Positifs (FP)  : 232215 (Fausses alarmes)
--------------------------------------------------
Précision de la séquence  = 1.02%
Recall de la séquence     = 80.72%
F1_score de la séquence   = 2.02%

Test de Local Thresholding par bandes avec la Métrique Mediane

RÉSULTATS GLOBAUX - SÉQUENCE : type : VGA sequence : sequence_1 dyn : low dyn with columns 3
Total Vrais Positifs (VP) : 2853
Total Faux Négatifs (FN)  : 114 (Oublis)
Total Faux Positifs (FP)  : 12517 (Fausses alarmes)
--------------------------------------------------
Précision de la séquence  = 18.56%
Recall de la séquence     = 96.16%
F1_score de la séquence   = 31.12%

Test de Local Thresholding par bandes avec la Métrique Moyenne

RÉSULTATS GLOBAUX - SÉQUENCE : type : VGA sequence : seq

(0.3626303739506487, 0.9609032692955848, 0.5265490811709299)

## Optimisation des params pour local thresholding with band for metrique Median and in col 3 of seq 1

In [43]:
import optuna

In [44]:
# ==========================================
# 1. LA FONCTION OBJECTIF POUR OPTUNA
# ==========================================
def objective(trial):
    """ 
    cette fonction suggère des val de param à tester -> donne le f1_score avec ce param 
    """
    # 1. Optuna suggère les paramètres à tester pour cet essai
    taille_fenetre_test = trial.suggest_int("taille_fenetre", 10, 150)
    facteur_std_test = trial.suggest_float("facteur_std", 1.0, 4.0)
    num_bandes_test = trial.suggest_int("num_bandes", 1, 10)
    
    # 2. On prépare notre fonction de détection avec ces nouveaux paramètres
    # On utilise partial pour "geler" les paramètres sans exécuter la fonction
    methode_test = partial(
        local_threshold_bandes, 
        taille_fenetre=taille_fenetre_test,
        facteur_std=facteur_std_test,
        num_bandes=num_bandes_test,
        metrique='Mediane',
        visual_graph=False # Surtout pas de graphes pendant l'optimisation !
    )
    
    # 3. On appelle TA fonction d'évaluation (en mode silencieux)
    precision, recall, f1_score = evaluate_sequence_detection(
        methode_detection=methode_test,
        folder='train',
        img_type='VGA',
        sequence='sequence_1',
        dyn='low dyn with columns 3',
        update=False # <-- On cache les prints pour ne pas polluer le terminal
    )
    
    # 4. On renvoie uniquement le F1-score à Optuna car c'est ce qu'il doit maximiser
    return f1_score


# ==========================================
# 2. LANCEMENT DE L'OPTIMISATION
# ==========================================
if __name__ == "__main__":
    print("🚀 Lancement de l'optimisation Bayésienne avec Optuna...")
    
    # On crée l'étude en demandant de maximiser la valeur renvoyée (le f1_score)
    study = optuna.create_study(direction="maximize")
    
    # On lance 50 essais (tu peux monter à 100 ou 200 si ça va vite)
    study.optimize(objective, n_trials=50)

    # ==========================================
    # 3. AFFICHAGE DU RÉSULTAT FINAL
    # ==========================================
    print("\n" + 50*"🌟")
    print("OPTIMISATION TERMINÉE")
    print(50*"🌟")
    print(f"Meilleur F1-Score atteint : {study.best_value * 100:.2f}%")
    print("Paramètres parfaits pour ce score :")
    for cle, valeur in study.best_params.items():
        print(f"  -> {cle} : {valeur}")

[I 2026-05-29 15:41:58,718] A new study created in memory with name: no-name-7158ca5d-75a9-45c2-8220-00b0fec7f2c4


🚀 Lancement de l'optimisation Bayésienne avec Optuna...


[I 2026-05-29 15:42:04,147] Trial 0 finished with value: 0.6881588624662908 and parameters: {'taille_fenetre': 75, 'facteur_std': 1.3347533859318066, 'num_bandes': 4}. Best is trial 0 with value: 0.6881588624662908.
[I 2026-05-29 15:42:09,245] Trial 1 finished with value: 0.6417944609750514 and parameters: {'taille_fenetre': 18, 'facteur_std': 3.50451516556664, 'num_bandes': 2}. Best is trial 0 with value: 0.6881588624662908.
[I 2026-05-29 15:42:14,683] Trial 2 finished with value: 0.6809500489715965 and parameters: {'taille_fenetre': 80, 'facteur_std': 1.8493224536783721, 'num_bandes': 8}. Best is trial 0 with value: 0.6881588624662908.
[I 2026-05-29 15:42:19,565] Trial 3 finished with value: 0.9163845633039946 and parameters: {'taille_fenetre': 37, 'facteur_std': 2.367835591965206, 'num_bandes': 10}. Best is trial 3 with value: 0.9163845633039946.
[I 2026-05-29 15:42:24,823] Trial 4 finished with value: 0.9483153995413653 and parameters: {'taille_fenetre': 124, 'facteur_std': 2.43964


🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
OPTIMISATION TERMINÉE
🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
Meilleur F1-Score atteint : 95.26%
Paramètres parfaits pour ce score :
  -> taille_fenetre : 25
  -> facteur_std : 2.122570050763343
  -> num_bandes : 4


### Optimisation des params pour local thresholding with band for metrique Moyenne and in col 3 of seq 1

In [45]:
# ==========================================
# 1. LA FONCTION OBJECTIF POUR OPTUNA
# ==========================================
def objective(trial):
    """ 
    cette fonction suggère des val de param à tester -> donne le f1_score avec ce param 
    """
    # 1. Optuna suggère les paramètres à tester pour cet essai
    taille_fenetre_test = trial.suggest_int("taille_fenetre", 10, 150)
    facteur_std_test = trial.suggest_float("facteur_std", 1.0, 4.0)
    num_bandes_test = trial.suggest_int("num_bandes", 1, 10)
    
    # 2. On prépare notre fonction de détection avec ces nouveaux paramètres
    # On utilise partial pour "geler" les paramètres sans exécuter la fonction
    methode_test = partial(
        local_threshold_bandes, 
        taille_fenetre=taille_fenetre_test,
        facteur_std=facteur_std_test,
        num_bandes=num_bandes_test,
        metrique='Moyenne',
        visual_graph=False # Surtout pas de graphes pendant l'optimisation !
    )
    
    # 3. On appelle TA fonction d'évaluation (en mode silencieux)
    precision, recall, f1_score = evaluate_sequence_detection(
        methode_detection=methode_test,
        folder='train',
        img_type='VGA',
        sequence='sequence_1',
        dyn='low dyn with columns 3',
        update=False # <-- On cache les prints pour ne pas polluer le terminal
    )
    
    # 4. On renvoie uniquement le F1-score à Optuna car c'est ce qu'il doit maximiser
    return f1_score


# ==========================================
# 2. LANCEMENT DE L'OPTIMISATION
# ==========================================
if __name__ == "__main__":
    print("🚀 Lancement de l'optimisation Bayésienne avec Optuna...")
    
    # On crée l'étude en demandant de maximiser la valeur renvoyée (le f1_score)
    study = optuna.create_study(direction="maximize")
    
    # On lance 50 essais (tu peux monter à 100 ou 200 si ça va vite)
    study.optimize(objective, n_trials=50)

    # ==========================================
    # 3. AFFICHAGE DU RÉSULTAT FINAL
    # ==========================================
    print("\n" + 50*"🌟")
    print("OPTIMISATION TERMINÉE")
    print(50*"🌟")
    print(f"Meilleur F1-Score atteint : {study.best_value * 100:.2f}%")
    print("Paramètres parfaits pour ce score :")
    for cle, valeur in study.best_params.items():
        print(f"  -> {cle} : {valeur}")

[I 2026-05-29 15:54:05,451] A new study created in memory with name: no-name-4c5f5c5f-2e5d-4afc-be70-74769205a71f


🚀 Lancement de l'optimisation Bayésienne avec Optuna...


[I 2026-05-29 15:54:09,733] Trial 0 finished with value: 0.3534261977637579 and parameters: {'taille_fenetre': 131, 'facteur_std': 1.3856330036305051, 'num_bandes': 7}. Best is trial 0 with value: 0.3534261977637579.
[I 2026-05-29 15:54:13,594] Trial 1 finished with value: 0.8108216432865731 and parameters: {'taille_fenetre': 150, 'facteur_std': 2.400884328141485, 'num_bandes': 1}. Best is trial 1 with value: 0.8108216432865731.
[I 2026-05-29 15:54:17,117] Trial 2 finished with value: 0.6632124352331606 and parameters: {'taille_fenetre': 112, 'facteur_std': 3.215308114830308, 'num_bandes': 1}. Best is trial 1 with value: 0.8108216432865731.
[I 2026-05-29 15:54:20,498] Trial 3 finished with value: 0.9457254345512593 and parameters: {'taille_fenetre': 16, 'facteur_std': 2.6707931633148547, 'num_bandes': 10}. Best is trial 3 with value: 0.9457254345512593.
[I 2026-05-29 15:54:23,807] Trial 4 finished with value: 0.9560516219044297 and parameters: {'taille_fenetre': 108, 'facteur_std': 1.9


🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
OPTIMISATION TERMINÉE
🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
Meilleur F1-Score atteint : 96.57%
Paramètres parfaits pour ce score :
  -> taille_fenetre : 36
  -> facteur_std : 1.4439935316800787
  -> num_bandes : 3


### Optimisation des params pour local thresholding with band for metrique Median and in col 3 of seq 2

In [46]:
# ==========================================
# 1. LA FONCTION OBJECTIF POUR OPTUNA
# ==========================================
def objective(trial):
    """ 
    cette fonction suggère des val de param à tester -> donne le f1_score avec ce param 
    """
    # 1. Optuna suggère les paramètres à tester pour cet essai
    taille_fenetre_test = trial.suggest_int("taille_fenetre", 10, 150)
    facteur_std_test = trial.suggest_float("facteur_std", 1.0, 4.0)
    num_bandes_test = trial.suggest_int("num_bandes", 1, 10)
    
    # 2. On prépare notre fonction de détection avec ces nouveaux paramètres
    # On utilise partial pour "geler" les paramètres sans exécuter la fonction
    methode_test = partial(
        local_threshold_bandes, 
        taille_fenetre=taille_fenetre_test,
        facteur_std=facteur_std_test,
        num_bandes=num_bandes_test,
        metrique='Moyenne',
        visual_graph=False # Surtout pas de graphes pendant l'optimisation !
    )
    
    # 3. On appelle TA fonction d'évaluation (en mode silencieux)
    precision, recall, f1_score = evaluate_sequence_detection(
        methode_detection=methode_test,
        folder='train',
        img_type='VGA',
        sequence='sequence_2',
        dyn='low dyn with columns 3',
        update=False # <-- On cache les prints pour ne pas polluer le terminal
    )
    
    # 4. On renvoie uniquement le F1-score à Optuna car c'est ce qu'il doit maximiser
    return f1_score


# ==========================================
# 2. LANCEMENT DE L'OPTIMISATION
# ==========================================
if __name__ == "__main__":
    print("🚀 Lancement de l'optimisation Bayésienne avec Optuna...")
    
    # On crée l'étude en demandant de maximiser la valeur renvoyée (le f1_score)
    study = optuna.create_study(direction="maximize")
    
    # On lance 50 essais (tu peux monter à 100 ou 200 si ça va vite)
    study.optimize(objective, n_trials=50)

    # ==========================================
    # 3. AFFICHAGE DU RÉSULTAT FINAL
    # ==========================================
    print("\n" + 50*"🌟")
    print("OPTIMISATION TERMINÉE")
    print(50*"🌟")
    print(f"Meilleur F1-Score atteint : {study.best_value * 100:.2f}%")
    print("Paramètres parfaits pour ce score :")
    for cle, valeur in study.best_params.items():
        print(f"  -> {cle} : {valeur}")

[I 2026-05-29 16:15:56,882] A new study created in memory with name: no-name-53b52876-e893-44ed-80a1-933f4ced869e


🚀 Lancement de l'optimisation Bayésienne avec Optuna...


[I 2026-05-29 16:16:04,086] Trial 0 finished with value: 0.06489978035097924 and parameters: {'taille_fenetre': 126, 'facteur_std': 1.154386302122205, 'num_bandes': 3}. Best is trial 0 with value: 0.06489978035097924.
[I 2026-05-29 16:16:10,496] Trial 1 finished with value: 0.372669604377512 and parameters: {'taille_fenetre': 21, 'facteur_std': 2.5061703550552785, 'num_bandes': 9}. Best is trial 1 with value: 0.372669604377512.
[I 2026-05-29 16:16:16,688] Trial 2 finished with value: 0.4584742115940247 and parameters: {'taille_fenetre': 87, 'facteur_std': 2.951863085448963, 'num_bandes': 10}. Best is trial 2 with value: 0.4584742115940247.
[I 2026-05-29 16:16:24,791] Trial 3 finished with value: 0.0368146553713564 and parameters: {'taille_fenetre': 149, 'facteur_std': 1.0777191041643062, 'num_bandes': 4}. Best is trial 2 with value: 0.4584742115940247.
[I 2026-05-29 16:16:35,166] Trial 4 finished with value: 0.11607105362617058 and parameters: {'taille_fenetre': 79, 'facteur_std': 1.68


🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
OPTIMISATION TERMINÉE
🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
Meilleur F1-Score atteint : 85.45%
Paramètres parfaits pour ce score :
  -> taille_fenetre : 16
  -> facteur_std : 3.420181549013675
  -> num_bandes : 4


### Optimisation des params pour CFAR_band RG in col 3 of seq 2

In [48]:
# ==========================================
# 1. LA FONCTION OBJECTIF POUR OPTUNA
# ==========================================
def objective(trial):
    """ 
    cette fonction suggère des val de param à tester -> donne le f1_score avec ce param 
    """
    # 1. Optuna suggère les paramètres à tester pour cet essai
    hauteur_bande = trial.suggest_int("hauteur_bande", 10, 150)
    train_cells = trial.suggest_float("train_cells", 4, 10)
    guard_cells = trial.suggest_int("guard_cells", 2, 5)
    multiplicateur_rupture = trial.suggest_float("multiplicateur_rupture", 2.0, 5.0)    

    # 2. On prépare notre fonction de détection avec ces nouveaux paramètres
    # On utilise partial pour "geler" les paramètres sans exécuter la fonction
    methode_test = partial(
        local_threshold_bandes, 
        hauteur_bande=hauteur_bande,
        train_cells=train_cells,
        guard_cells=guard_cells,
        multiplicateur_rupture=multiplicateur_rupture,
        metrique='Moyenne',
        visual_graph=False # Surtout pas de graphes pendant l'optimisation !
    )
    
    # 3. On appelle TA fonction d'évaluation (en mode silencieux)
    precision, recall, f1_score = detecter_et_mesurer_defauts_complet(
        methode_detection=methode_test,
        folder='train',
        img_type='VGA',
        sequence='sequence_2',
        dyn='low dyn with columns 3',
        update=False # <-- On cache les prints pour ne pas polluer le terminal
    )
    
    # 4. On renvoie uniquement le F1-score à Optuna car c'est ce qu'il doit maximiser
    return f1_score


# ==========================================
# 2. LANCEMENT DE L'OPTIMISATION
# ==========================================
if __name__ == "__main__":
    print("🚀 Lancement de l'optimisation Bayésienne avec Optuna...")
    
    # On crée l'étude en demandant de maximiser la valeur renvoyée (le f1_score)
    study = optuna.create_study(direction="maximize")
    
    # On lance 50 essais (tu peux monter à 100 ou 200 si ça va vite)
    study.optimize(objective, n_trials=50)

    # ==========================================
    # 3. AFFICHAGE DU RÉSULTAT FINAL
    # ==========================================
    print("\n" + 50*"🌟")
    print("OPTIMISATION TERMINÉE")
    print(50*"🌟")
    print(f"Meilleur F1-Score atteint : {study.best_value * 100:.2f}%")
    print("Paramètres parfaits pour ce score :")
    for cle, valeur in study.best_params.items():
        print(f"  -> {cle} : {valeur}")

[I 2026-05-29 16:23:13,384] A new study created in memory with name: no-name-a4d2393d-4be7-4203-86a6-b4bcc78b0cd7
[W 2026-05-29 16:23:13,396] Trial 0 failed with parameters: {'hauteur_bande': 126, 'train_cells': 5.8753639554201085, 'guard_cells': 4, 'multiplicateur_rupture': 2.027647763607237} because of the following error: TypeError("detecter_et_mesurer_defauts_complet() got an unexpected keyword argument 'methode_detection'").
Traceback (most recent call last):
  File "/opt/anaconda3/envs/phelma/lib/python3.10/site-packages/optuna/study/_optimize.py", line 205, in _run_trial
    value_or_values = func(trial)
  File "/var/folders/ty/mz254c797ml4f8jydylf3h6c0000gn/T/ipykernel_96067/3395951669.py", line 27, in objective
    precision, recall, f1_score = detecter_et_mesurer_defauts_complet(
TypeError: detecter_et_mesurer_defauts_complet() got an unexpected keyword argument 'methode_detection'
[W 2026-05-29 16:23:13,397] Trial 0 failed with value None.


🚀 Lancement de l'optimisation Bayésienne avec Optuna...


TypeError: detecter_et_mesurer_defauts_complet() got an unexpected keyword argument 'methode_detection'